# 💼 Job Application AI Agent\n### Built with Groq (`openai/gpt-oss-120b`), LangChain, HuggingFace, FAISS Vector DB & Gradio\n\nThis notebook follows the complete step-by-step pipeline:\n1. **Install packages:** `groq`, `gradio`, `langchain`, `langchain-groq`, `langchain-huggingface`, `faiss-cpu`, `pypdf`, `reportlab`\n2. **Create PDF resume:** Generate `sample_resume.pdf` with ReportLab\n3. **Configure Groq API:** Initialize Groq client with your key\n4. **Test Groq:** Verify high-speed inference with `openai/gpt-oss-120b`\n5. **Load & Chunk PDF:** PyPDFLoader + RecursiveCharacterTextSplitter\n6. **Embeddings & Vector Store:** HuggingFace `all-MiniLM-L6-v2` + FAISS\n7. **Create Tools:** Resume RAG tool, Salary & Budget calculator tool, Tavily market search\n8. **Launch Gradio Web App:** Interactive UI with live public URL (`share=True`)

In [ ]:
!pip install -q groq gradio langchain langchain-groq langchain-community langchain-huggingface sentence-transformers faiss-cpu pypdf reportlab tavily-python

In [ ]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

pdf_filename = "sample_resume.pdf"

c = canvas.Canvas(pdf_filename, pagesize=letter)
c.setFont("Helvetica-Bold", 16)
c.drawString(50, 750, "Alexander Chen - Senior Full Stack & AI Engineer")

c.setFont("Helvetica", 10)
c.drawString(50, 735, "Email: alex.chen@example.com | Phone: (555) 234-5678 | San Francisco, CA")
c.line(50, 725, 550, 725)

c.setFont("Helvetica-Bold", 12)
c.drawString(50, 705, "PROFESSIONAL SUMMARY")
c.setFont("Helvetica", 10)
c.drawString(50, 690, "Senior Full Stack & AI Engineer with 6+ years experience in Python, LangChain, Groq,")
c.drawString(50, 675, "React, TypeScript, Vector Databases (FAISS), and high-throughput cloud microservices.")

c.setFont("Helvetica-Bold", 12)
c.drawString(50, 650, "TECHNICAL SKILLS")
c.setFont("Helvetica", 10)
c.drawString(50, 635, "- Languages & Frameworks: Python, TypeScript, React, Next.js, FastAPI, Node.js")
c.drawString(50, 620, "- AI & RAG: LangChain, Groq API, HuggingFace, FAISS, Vector Search, Prompt Engineering")
c.drawString(50, 605, "- Databases & Cloud: PostgreSQL, Redis, Docker, AWS, GCP, CI/CD pipelines")

c.setFont("Helvetica-Bold", 12)
c.drawString(50, 580, "EXPERIENCE")
c.setFont("Helvetica-Bold", 11)
c.drawString(50, 565, "Senior Full Stack & AI Engineer - InnovateAI Labs (2022 - Present)")
c.setFont("Helvetica", 10)
c.drawString(50, 550, "- Architected enterprise LangChain + Groq RAG pipeline, cutting document query time by 80%.")
c.drawString(50, 535, "- Led front-end and back-end integration in React and FastAPI serving 2M+ active users.")
c.drawString(50, 520, "- Boosted vector retrieval precision from 72% to 94.6% using hybrid chunking strategies.")

c.setFont("Helvetica-Bold", 11)
c.drawString(50, 495, "Software Engineer - CloudScale Systems (2019 - 2022)")
c.setFont("Helvetica", 10)
c.drawString(50, 480, "- Built microservices handling 30,000+ requests/min in Python and PostgreSQL.")
c.drawString(50, 465, "- Reduced p95 response time by 120ms with Redis caching and query indexing.")

c.setFont("Helvetica-Bold", 12)
c.drawString(50, 440, "EDUCATION")
c.setFont("Helvetica", 10)
c.drawString(50, 425, "B.S. in Computer Science - University of California, Berkeley (2015 - 2019)")

c.save()
print(f"✅ Created resume PDF: {pdf_filename} ({os.path.getsize(pdf_filename)} bytes)")

In [ ]:
import os
import groq

# Configure API Keys
os.environ["GROQ_API_KEY"] = "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"
os.environ["TAVILY_API_KEY"] = "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"

# Initialize Groq client
client = groq.Groq(api_key=os.environ["GROQ_API_KEY"])
print("✅ Groq client initialized successfully with key: " + os.environ["GROQ_API_KEY"][:8] + "...")

In [ ]:
# Test Groq Model Response (Verified models for your Groq API key)
test_models = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.8-27b"]
active_model = None

for m in test_models:
    try:
        response = client.chat.completions.create(
            model=m,
            messages=[
                {"role": "system", "content": "You are the Job Application Agent. Reply in 1 concise sentence confirming readiness."},
                {"role": "user", "content": "Are you online and ready to assist with job applications?"}
            ],
            temperature=0.2,
            max_tokens=60
        )
        active_model = m
        print(f"✅ Groq Model Connected: '{m}'")
        print("🤖 Agent Response:", response.choices[0].message.content)
        break
    except Exception as e:
        print(f"⚠️ Model '{m}' check notice: {str(e)[:90]}... (trying next)")

if not active_model:
    raise RuntimeError("Could not connect to any Groq model. Check API key.")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

# 1. Load PDF using LangChain
pdf_path = "sample_resume.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"📄 Loaded {len(documents)} page(s) from '{pdf_path}'")

# 2. Split into chunks using RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"✂️ Split resume into {len(chunks)} text chunks")

# 3. Create HuggingFace Embeddings (free, runs locally on CPU/GPU)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("🧠 HuggingFace Embeddings model loaded successfully")

# 4. Build FAISS Vector Database
vector_db = FAISS.from_documents(chunks, embeddings)
print("📦 FAISS Vector Database built & indexed successfully")

# 5. Create Retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 2})
print("🔍 Retriever configured (k=2 nearest neighbors)")

# 6. Test RAG Retrieval
test_query = "What AI and vector database skills does the candidate have?"
retrieved_docs = retriever.invoke(test_query)

print(f"\n🎯 RAG Test Query: '{test_query}'")
for idx, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk #{idx} ---")
    print(doc.page_content.strip())

In [ ]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from tavily import TavilyClient

# 1. Connect LangChain to Groq using the verified active model
llm = ChatGroq(
    model_name="openai/gpt-oss-120b",
    groq_api_key=os.environ["GROQ_API_KEY"],
    temperature=0.3
)
print("🔗 LangChain ChatGroq connected with model 'openai/gpt-oss-120b'!")

# 2. Tool 1: Resume RAG Function & Tool
def query_resume_data(query: str) -> str:
    """Searches the candidate resume for relevant text chunks."""
    results = retriever.invoke(query)
    if not results:
        return "No relevant resume data found for this query."
    return "\n---\n".join([doc.page_content for doc in results])

@tool
def resume_rag_tool(query: str) -> str:
    """Searches the candidate's resume PDF for specific skills, past work experience, achievements, and education."""
    return query_resume_data(query)

# 3. Tool 2: Budget & Compensation Function & Tool
def calculate_compensation_report(role_title: str, experience_years: float, location: str = "Remote") -> str:
    """Calculates salary bands, target offer, and negotiation tactics."""
    exp = float(experience_years)
    base_salaries = {
        "full stack": 125000,
        "frontend": 115000,
        "backend": 125000,
        "ai": 145000,
        "machine learning": 145000,
        "data": 120000,
        "devops": 130000,
        "software": 120000
    }
    
    matched_base = 120000
    for key, val in base_salaries.items():
        if key in role_title.lower():
            matched_base = val
            break
            
    calculated_base = matched_base * (1 + (exp * 0.075))
    min_sal = round(calculated_base * 0.90, -3)
    target_sal = round(calculated_base, -3)
    max_sal = round(calculated_base * 1.15, -3)
    sign_bonus = round(calculated_base * 0.08, -3)
    equity = "0.10% - 0.35%" if exp >= 5 else "0.02% - 0.10%"
    
    return f"""
💼 COMPENSATION & BUDGET REPORT:
• Role: {role_title} ({exp} yrs experience | {location})
• Base Salary Range: ${int(min_sal):,} - ${int(max_sal):,} USD
• Recommended Target Offer: ${int(target_sal):,} USD
• Suggested Signing Bonus: ${int(sign_bonus):,} USD
• Equity Band: {equity}
• Tactical Negotiation Tip: Anchor at ${int(max_sal):,} USD by highlighting quantified metrics from previous projects.
"""

@tool
def budget_and_salary_tool(role_title: str, experience_years: float, location: str = "Remote") -> str:
    """Calculates competitive salary budget, market compensation bands, signing bonuses, and negotiation tactics for a given job role and years of experience."""
    return calculate_compensation_report(role_title, experience_years, location)

# 4. Tool 3: Live Market Search via Tavily
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def search_live_market(query: str) -> str:
    """Searches live job listings with Tavily."""
    try:
        search_res = tavily_client.search(query=query + " software engineering jobs requirements", max_results=3)
        snippets = [f"• {item['title']}: {item['content'][:180]}... ({item['url']})" for item in search_res.get("results", [])]
        return "\n".join(snippets) if snippets else "No web results found."
    except Exception as e:
        return f"Market search notice: {e}"

@tool
def live_job_market_tool(query: str) -> str:
    """Searches live job openings, company requirements, and tech stacks using Tavily web search."""
    return search_live_market(query)

print("🛠️ Registered Tools: [resume_rag_tool, budget_and_salary_tool, live_job_market_tool]")

In [ ]:
def run_job_application_agent(user_message: str, history=None) -> str:
    """Processes user queries about job applications, analyzes resumes, and calculates compensation."""
    user_lower = str(user_message).lower()
    
    # 1. Check if user is asking for salary / budget / compensation
    budget_keywords = ["salary", "budget", "compensation", "pay", "offer", "negotiat", "worth", "package"]
    if any(k in user_lower for k in budget_keywords):
        exp = 6.0
        role = "Senior Full Stack & AI Engineer"
        budget_info = calculate_compensation_report(role, exp, "Remote")
        
        prompt = f"""
System: You are an expert Job Application Agent and Executive Career Coach.
User Question: {user_message}
Calculated Compensation Data:
{budget_info}

Provide a comprehensive, professional response advising the candidate or hiring manager. Include exact numbers, offer benchmarks, and negotiation strategies.
"""
        response = llm.invoke(prompt)
        return response.content
        
    # 2. Check if user is asking about resume / candidate background / fit / skills
    resume_keywords = ["resume", "experience", "skill", "background", "project", "work", "fit", "cover letter", "qualification"]
    if any(k in user_lower for k in resume_keywords) or len(user_message.strip()) < 10:
        resume_data = query_resume_data(user_message)
        prompt = f"""
System: You are the Job Application Agent. You have access to the candidate's verified resume extracted via RAG.
Candidate Resume Context:
{resume_data}

User Request: {user_message}

Answer with precision. If writing a pitch or cover letter, ground every bullet in the candidate's real experience. Highlight quantified accomplishments.
"""
        response = llm.invoke(prompt)
        return response.content
        
    # 3. General Job Search or Application Advisory
    market_data = search_live_market(user_message)
    prompt = f"""
System: You are the Job Application Agent.
Market Search Context:
{market_data}

User Question: {user_message}

Provide actionable career strategy, interview preparation, and job application guidance.
"""
    response = llm.invoke(prompt)
    return response.content

# Test Run
sample_response = run_job_application_agent("What are my top 3 technical strengths from my resume, and what salary should I target for a Senior AI Engineer role?")
print("🤖 Agent Test Run Output:\n")
print(sample_response[:500] + "...")

In [ ]:
import gradio as gr

def gradio_chat(message, history):
    msg_str = message.get("text", "") if isinstance(message, dict) else str(message)
    return run_job_application_agent(msg_str, history)

def gradio_budget_calc(role, exp_years, location):
    return calculate_compensation_report(role, float(exp_years), location)

def gradio_rag_search(query):
    return query_resume_data(query)

# Build custom multi-tab Gradio UI
with gr.Blocks(theme=gr.themes.Soft(), title="Job Application Agent") as demo:
    gr.Markdown("""
    # 💼 Job Application AI Agent
    ### Powered by Groq (openai/gpt-oss-120b), LangChain, HuggingFace & FAISS
    Ask questions about candidate credentials, draft tailored pitch letters, evaluate job fit, and compute competitive salary compensation packages.
    """)
    
    with gr.Tab("💬 Job Agent Chat"):
        chatbot = gr.ChatInterface(
            fn=gradio_chat,
            examples=[
                "Write a compelling 3-paragraph cover letter for a Senior AI & Full Stack role.",
                "What are my key skills and achievements from my resume?",
                "What salary and signing bonus should I ask for a Senior Engineer with 6 years experience?",
                "How does my background fit a Lead RAG Architecture role?"
            ],
            title=None
        )
        
    with gr.Tab("💰 Budget & Salary Calculator"):
        gr.Markdown("### Calculate Market Compensation & Budget Bands")
        with gr.Row():
            role_input = gr.Textbox(label="Job Role Title", value="Senior Full Stack & AI Engineer")
            exp_input = gr.Slider(minimum=0, maximum=20, value=6, step=0.5, label="Years of Experience")
            loc_input = gr.Dropdown(choices=["Remote (US)", "San Francisco / Bay Area", "New York", "Austin / Seattle", "Global Remote"], value="Remote (US)", label="Location Tier")
        calc_btn = gr.Button("Calculate Compensation Package", variant="primary")
        budget_output = gr.Textbox(label="Compensation & Negotiation Report", lines=8)
        calc_btn.click(fn=gradio_budget_calc, inputs=[role_input, exp_input, loc_input], outputs=budget_output)
        
    with gr.Tab("📄 Resume RAG Explorer"):
        gr.Markdown("### Direct Semantic Search over Resume Embeddings (FAISS)")
        with gr.Row():
            rag_query = gr.Textbox(label="Resume Search Query", placeholder="e.g. LangChain experience, React projects, education", scale=4)
            rag_btn = gr.Button("Search Chunks", variant="primary", scale=1)
        rag_output = gr.Textbox(label="Retrieved Document Chunks", lines=6)
        rag_btn.click(fn=gradio_rag_search, inputs=[rag_query], outputs=rag_output)

# Launch Gradio with share=True to generate a public link
demo.launch(share=True, debug=True)